# Autocovariance

Autocovariance $\gamma(h)=\mathrm{Cov}(X_t,X_{t+h})$ keeps the squared units of the series; dividing by $\gamma(0)$ gives the ACF. This notebook covers the two sample-denominator conventions, the AR(1) and MA(1) theoretical values, and how the estimate behaves at large lags.

Companion note: [autocovariance_function.md](../../notes/time_series/autocovariance_function.md).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 3.5)

x = np.array([1., 2., 4., 3.])
z = x - x.mean()
n = len(x)
g0 = np.sum(z * z) / n
g1_n = np.sum(z[:-1] * z[1:]) / n          # divisor n
g1_pairs = np.sum(z[:-1] * z[1:]) / (n - 1)  # divisor n-h
print('gamma0 =', g0, ' gamma1 (n) =', g1_n, ' gamma1 (n-h) =', g1_pairs)
print('rho1 =', g1_n / g0)

gamma0 = 1.25  gamma1 (n) = 0.1875  gamma1 (n-h) = 0.25
rho1 = 0.15


Both conventions are in use; dividing by $n$ guarantees a non-negative-definite sequence. Below, the AR(1) values $\gamma(0)=\sigma^2/(1-\phi^2)$ and $\gamma(h)=\phi^h\gamma(0)$ for $\phi=0.7$ are compared with a long simulation.

In [2]:
def sample_acov(x, h, divisor='n'):
    z = np.asarray(x) - np.mean(x)
    d = len(z) if divisor == 'n' else len(z) - h
    return np.sum(z[:len(z)-h] * z[h:]) / d

phi, sigma2, N = 0.7, 1.0, 20000
e = rng.normal(scale=np.sqrt(sigma2), size=N + 200)
y = np.zeros(N + 200)
for t in range(1, N + 200):
    y[t] = phi * y[t-1] + e[t]
y = y[200:]
g0_th = sigma2 / (1 - phi**2)
for h in range(4):
    print(f'h={h}: theory={phi**h * g0_th:.4f}  sample={sample_acov(y, h):.4f}')

h=0: theory=1.9608  sample=2.0141
h=1: theory=1.3725  sample=1.4193
h=2: theory=0.9608  sample=0.9888
h=3: theory=0.6725  sample=0.6866


Theory ($1.9608$, $1.3725$, $0.9608$, ...) and simulation agree closely. For an MA(1), $\gamma(0)=\sigma^2(1+\theta^2)$, $\gamma(1)=\sigma^2\theta$, and zero beyond.

In [3]:
theta = 0.6
e = rng.normal(size=N + 1)
m = e[1:] + theta * e[:-1]
print('theory  :', [round(1 + theta**2, 3), theta, 0.0, 0.0])
print('sample  :', [round(sample_acov(m, h), 3) for h in range(4)])
print('rho(1) theory theta/(1+theta^2) =', round(theta / (1 + theta**2), 3),
      ' sample =', round(sample_acov(m, 1) / sample_acov(m, 0), 3))

theory  : [1.36, 0.6, 0.0, 0.0]
sample  : [np.float64(1.363), np.float64(0.592), np.float64(-0.012), np.float64(0.001)]
rho(1) theory theta/(1+theta^2) = 0.441  sample = 0.435


At large lags few pairs enter the sum, so estimates are noisy. With a short series, the two denominators visibly diverge.

In [4]:
short = y[:60]
H = 40
hs = np.arange(H)
gn = [sample_acov(short, h, 'n') for h in hs]
gp = [sample_acov(short, h, 'pairs') for h in hs]
plt.plot(hs, g0_th * phi ** hs, 'k-', label='theory')
plt.plot(hs, gn, 'o-', label='divisor n')
plt.plot(hs, gp, 's--', label='divisor n-h')
plt.xlabel('lag h'); plt.ylabel('autocovariance'); plt.legend(); plt.show()

/tmp/ipykernel_78432/4238903260.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('lag h'); plt.ylabel('autocovariance'); plt.legend(); plt.show()
